# Team Challenge — Exploración y entrenamiento del modelo

Este notebook documenta la fase de **exploración de datos y entrenamiento** del modelo
que luego se sirve mediante la API (`app/main.py`). No forma parte del despliegue en sí
(Render corre `model/train_model.py`, no este notebook) — es el "cuaderno de trabajo"
del equipo antes de pasar el modelo a producción.

**Rol sugerido:** Persona A (Modelo / Datos), revisado por Persona B antes de fijar
el contrato de entrada de la API.


## 1. Carga de datos

Usamos el dataset **Iris** (abierto, incluido en scikit-learn) como ejemplo de modelo a desplegar.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

data = load_iris()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
df["target_name"] = df["target"].map(dict(enumerate(data.target_names)))
df.head()


In [ ]:
df.describe()


In [ ]:
df["target_name"].value_counts()


## 2. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


## 3. Entrenamiento del modelo

Probamos un `RandomForestClassifier` como baseline razonable para este dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))


In [ ]:
confusion_matrix(y_test, y_pred)


## 4. Importancia de variables

Útil para justificar en la presentación por qué el modelo predice lo que predice.

In [ ]:
importances = pd.Series(model.feature_importances_, index=data.feature_names)
importances.sort_values(ascending=False)


## 5. Guardar el modelo para la API

Este es el mismo paso que ejecuta `model/train_model.py` en el proyecto de despliegue.
Se guarda el modelo junto con los nombres de las clases para que el endpoint `/predict`
no dependa de tener `sklearn.datasets` disponible en producción.

In [ ]:
import joblib

bundle = {
    "model": model,
    "target_names": list(data.target_names),
    "feature_names": list(data.feature_names),
}
joblib.dump(bundle, "model.pkl")
print("Modelo guardado como model.pkl")


## 6. Probar el modelo cargado (sanity check)

Antes de copiar `model.pkl` al proyecto de la API, comprobamos que se carga y predice bien.

In [ ]:
loaded = joblib.load("model.pkl")
sample = [[5.1, 3.5, 1.4, 0.2]]  # debería ser 'setosa'

pred_idx = loaded["model"].predict(sample)[0]
print("Predicción:", loaded["target_names"][pred_idx])


## Siguientes pasos

1. Copiar `model.pkl` (o el script `train_model.py` equivalente) al repo del proyecto de despliegue.
2. Persona B integra el modelo en `app/main.py` (FastAPI) siguiendo el esquema `IrisInput` acordado.
3. Probar en local con `test_api.py` antes de hacer push a Render.
4. Ver `README.md` del proyecto de despliegue para el reparto de roles completo y la guía paso a paso.